In [ ]:
"""Exploratory Data Analysis on a Cars Dataset

This project uses a reproducible, synthetic automobile dataset created in this
file for learning and EDA demonstration. It is NOT the original UCI Auto MPG
records. Run: python Muthuselvam_R_Cars_EDA_Project.py
Dependencies: pandas, numpy, matplotlib, seaborn
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
OUTPUT_DIR = "cars_eda_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
sns.set_theme(style="whitegrid")

## 1. Create a reproducible automobile dataset

In [ ]:
n = 500
cylinders = rng.choice([3, 4, 5, 6, 8], size=n, p=[0.03, 0.48, 0.04, 0.25, 0.20])
displacement = np.clip(cylinders * rng.normal(37, 5, n), 55, 480)
horsepower = np.clip(displacement * rng.normal(0.43, 0.07, n), 45, 250)
weight = np.clip(1450 + displacement * 6.0 + cylinders * 95 + rng.normal(0, 280, n), 1500, 5200)
acceleration = np.clip(25 - horsepower * 0.045 + rng.normal(0, 2.2, n), 8, 25)
model_year = rng.integers(1970, 1983, n)

## Fuel efficiency decreases with weight/engine size and improves in later years.

In [ ]:
mpg = 52 - weight / 155 - horsepower / 16 + (model_year - 1970) * 0.62 + rng.normal(0, 3.2, n)
mpg = np.clip(mpg, 8, 48)
origin_code = rng.choice([1, 2, 3], size=n, p=[0.62, 0.18, 0.20])
brands = {1: ["Ford", "Chevrolet", "Dodge", "Buick", "Plymouth"],
          2: ["Volkswagen", "Volvo", "Audi", "Peugeot", "Fiat"],
          3: ["Toyota", "Honda", "Datsun", "Mazda", "Subaru"]}
car_names = [f"{rng.choice(brands[o])} Model-{i+1}" for i, o in enumerate(origin_code)]
raw = pd.DataFrame({
    "car_name": car_names, "mpg": mpg.round(1), "cylinders": cylinders,
    "displacement": displacement.round(1), "horsepower": horsepower.round(1),
    "weight": weight.round(0).astype(int), "acceleration": acceleration.round(1),
    "model_year": model_year, "origin": origin_code
})

## Add realistic data-quality issues so the cleaning workflow can be demonstrated.

In [ ]:
missing_rows = rng.choice(raw.index, size=12, replace=False)
raw.loc[missing_rows, "horsepower"] = np.nan
raw = pd.concat([raw, raw.sample(5, random_state=RANDOM_SEED)], ignore_index=True)

## One intentionally extreme record to demonstrate outlier detection.

In [ ]:
raw.loc[0, "weight"] = 7800
raw.loc[0, "horsepower"] = 390

print("=" * 65)
print("EXPLORATORY DATA ANALYSIS ON A CARS DATASET")
print("=" * 65)
print("Raw dataset shape:", raw.shape)
print("\\nSample rows:\\n", raw.head())
print("\\nData types:\\n", raw.dtypes)
print("\\nMissing values:\\n", raw.isnull().sum())
print("Duplicate rows:", raw.duplicated().sum())

## 2. Data quality checks and cleaning

In [ ]:
clean = raw.drop_duplicates().copy()
missing_before = clean.isnull().sum()

## Median imputation is used only for the numerical horsepower feature.

In [ ]:
clean["horsepower"] = clean["horsepower"].fillna(clean["horsepower"].median())

## Convert origin codes to meaningful labels.

In [ ]:
origin_labels = {1: "USA", 2: "Europe", 3: "Japan"}
clean["origin"] = clean["origin"].map(origin_labels).fillna("Unknown")
clean["model_year"] = clean["model_year"].astype(int)
print("\\nRows before cleaning:", len(raw))
print("Rows after removing duplicates:", len(clean))
print("Missing values before imputation (after duplicate removal):\\n", missing_before)
print("Missing values after cleaning:\\n", clean.isnull().sum())
print("\\nCleaned dataset information:\\n")
clean.info()

## 3. Descriptive statistics

In [ ]:
numeric_cols = clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = clean.select_dtypes(exclude=np.number).columns.tolist()
print("\\nNumerical descriptive statistics:\\n", clean[numeric_cols].describe().round(2))
print("\\nCategorical descriptive statistics:\\n", clean[categorical_cols].describe())
print("\\nOrigin counts:\\n", clean["origin"].value_counts())
print("\\nCylinder counts:\\n", clean["cylinders"].value_counts().sort_index())

## 4. Outlier identification using the IQR rule (flag only; do not delete).

In [ ]:
outlier_summary = {}
for col in ["mpg", "horsepower", "weight", "displacement"]:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((clean[col] < lower) | (clean[col] > upper)).sum())
    outlier_summary[col] = count
    print(f"Outliers in {col}: {count} (IQR bounds: {lower:.2f} to {upper:.2f})")

## 5. Visual EDA: six charts, with titles, labels and saved image files.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(clean["mpg"], bins=20, edgecolor="black", alpha=0.8)
plt.title("Distribution of Fuel Efficiency (MPG)")
plt.xlabel("Miles per Gallon (MPG)"); plt.ylabel("Number of Cars")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "01_mpg_distribution.png"), dpi=160); plt.close()

plt.figure(figsize=(8, 5))
clean["cylinders"].value_counts().sort_index().plot(kind="bar", edgecolor="black")
plt.title("Number of Cars by Cylinder Count")
plt.xlabel("Number of Cylinders"); plt.ylabel("Number of Cars")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "02_cylinder_counts.png"), dpi=160); plt.close()

plt.figure(figsize=(8, 5))
clean.boxplot(column="mpg", by="cylinders", grid=True)
plt.title("Fuel Efficiency Across Cylinder Groups")
plt.xlabel("Number of Cylinders"); plt.ylabel("Miles per Gallon (MPG)")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "03_mpg_by_cylinders.png"), dpi=160); plt.close()

plt.figure(figsize=(8, 5))
for label, group in clean.groupby("origin"):
    plt.scatter(group["weight"], group["mpg"], label=label, alpha=0.65)
plt.legend(title="Origin")
plt.title("Vehicle Weight vs Fuel Efficiency")
plt.xlabel("Vehicle Weight (pounds)"); plt.ylabel("Miles per Gallon (MPG)")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "04_weight_vs_mpg.png"), dpi=160); plt.close()

plt.figure(figsize=(8, 5))
for label, group in clean.groupby("cylinders"):
    plt.scatter(group["horsepower"], group["mpg"], label=str(label), alpha=0.65)
plt.legend(title="Cylinders")
plt.title("Horsepower vs Fuel Efficiency")
plt.xlabel("Horsepower"); plt.ylabel("Miles per Gallon (MPG)")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "05_horsepower_vs_mpg.png"), dpi=160); plt.close()

plt.figure(figsize=(9, 6))
correlation = clean[numeric_cols].corr(numeric_only=True)
image = plt.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(image, label="Correlation")
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha="right")
plt.yticks(range(len(correlation.index)), correlation.index)
for i in range(len(correlation.index)):
    for j in range(len(correlation.columns)):
        plt.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "06_correlation_heatmap.png"), dpi=160); plt.close()

## 6. Evidence-based findings generated from the cleaned dataset.

In [ ]:
weight_corr = clean["weight"].corr(clean["mpg"])
hp_corr = clean["horsepower"].corr(clean["mpg"])
mean_by_cyl = clean.groupby("cylinders")["mpg"].mean().sort_values(ascending=False)
mean_by_origin = clean.groupby("origin")["mpg"].mean().sort_values(ascending=False)
mean_by_year = clean.groupby("model_year")["mpg"].mean()
print("\\nKEY FINDINGS")
print(f"1. Correlation between weight and MPG: {weight_corr:.3f}.")
print(f"2. Correlation between horsepower and MPG: {hp_corr:.3f}.")
print(f"3. Highest average MPG cylinder group: {mean_by_cyl.index[0]} cylinders ({mean_by_cyl.iloc[0]:.2f} MPG).")
print(f"4. Highest average MPG origin group: {mean_by_origin.index[0]} ({mean_by_origin.iloc[0]:.2f} MPG).")
print(f"5. Average MPG changed from {mean_by_year.iloc[0]:.2f} in {mean_by_year.index[0]} to {mean_by_year.iloc[-1]:.2f} in {mean_by_year.index[-1]}.")
print("\\nSix charts saved in:", os.path.abspath(OUTPUT_DIR))
print("Note: This is synthetic educational data; findings describe this generated sample only.")